<a href="https://colab.research.google.com/github/niloofarmaani/Are-Binns-just-GNNs-/blob/main/BINN_vs_GNN_TCGA_comparison_UPDATED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TCGA Tumor vs Normal: BINN (official) vs GNN variants

This notebook compares:

- **Official BINN package** implementation (Hartman et al., `InfectionMedicineProteomics/BINN`)
- Layered **GNN-style baselines** on the same Reactome-derived hierarchy (depth-matched)
- A **native DAG** propagation baseline (no padding) using the same underlying graph

Key fairness constraints enforced here:
- Same train/validation split for every model
- Same input matrix **(same samples, same genes, same ordering)**
- GNN depth is **not allowed to exceed** the BINN depth


In [ ]:
import sys
from pathlib import Path
import os

PROJECT_ROOT = Path("/home/maani/niloo/binn_gnn_repo_ready/binn_gnn_repo_ready")

sys.path.append(str(PROJECT_ROOT / "src"))
os.environ["BINN_GNN_BASE"] = str(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Project root: /home/maani/niloo/binn_gnn_repo_ready/binn_gnn_repo_ready


In [ ]:
import pyarrow
import pandas as pd
print(pyarrow.__version__)

21.0.0


In [ ]:
# === 1) Imports + config ===
from pathlib import Path
import os, json, time
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

# Local package
from binn_gnn.graph.schedule import build_layered_schedule, build_depth_schedule
from binn_gnn.models.layered import GCNSharedD1, GATGateD1, VectorSharedD, DenseMLPBaseline, BINNExactD1
from binn_gnn.models.dag import DAGBINNExactD1

# Official BINN package
from binn import BINN

# -----------------
# Paths
# -----------------
BASE_DIR = Path(os.environ.get("BINN_GNN_BASE", ".")).resolve()
OUT_DIR  = BASE_DIR / "outputs"
RAW_GRAPH_DIR = OUT_DIR / "graph"
LAYERED_GRAPH_DIR = OUT_DIR / "graph_layered_binn"

EXPR_PARQUET = OUT_DIR / "expr_reactome_tcga_tumor_normal.parquet"
Y_CSV        = OUT_DIR / "y_tcga_tumor_normal.csv"

# -----------------
# Experiment config
# -----------------
SEED = 42
VAL_FRAC = 0.2

EPOCHS = 10          # increase for final runs
BATCH_SIZE = 64
EARLY_STOPPING = 10

# shared hyperparam grid (small by default)
GRID = {
    "lr": [1e-3, 3e-4],
    "weight_decay": [1e-5],
    "dropout": [0.0, 0.2],
    "d_vector": [16],   # only used by VectorSharedD
}

STANDARDIZE_X = True   # recommended (train-only fit)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)


/home/maani/miniconda3/envs/nilooenv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu


In [ ]:
# === 2) Verify artifacts exist (run notebook 01 if missing) ===
required = [
    EXPR_PARQUET, Y_CSV,
    RAW_GRAPH_DIR / "node_table.csv", RAW_GRAPH_DIR / "edge_index.pt", RAW_GRAPH_DIR / "edge_table.csv",
    LAYERED_GRAPH_DIR / "node_table_layered.csv", LAYERED_GRAPH_DIR / "edge_table_layered.csv", LAYERED_GRAPH_DIR / "edge_index_layered.pt",
]
missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files. Run notebooks/01_graph_preparation_refactored.ipynb first.\n"
        + "\n".join(missing)
    )
print("All required artifacts found.")


All required artifacts found.


In [ ]:
# === 3) Load dataset + define ONE fixed split (shared across all baselines) ===
expr = pd.read_parquet(EXPR_PARQUET)   # genes x samples
y_df = pd.read_csv(Y_CSV, index_col=0)

samples = y_df.index.astype(str).tolist()
y_np = y_df.iloc[:, 0].to_numpy(dtype=np.int64)

# Align expression columns to y order
expr = expr.loc[:, samples]

idx = np.arange(len(samples))
train_idx_np, val_idx_np = train_test_split(
    idx,
    test_size=VAL_FRAC,
    random_state=SEED,
    stratify=y_np
)

train_idx = torch.tensor(train_idx_np, dtype=torch.long)
val_idx   = torch.tensor(val_idx_np, dtype=torch.long)

print("samples:", len(samples), "genes:", expr.shape[0])
print("train:", len(train_idx), "val:", len(val_idx), "tumor_frac:", float(y_np.mean()))


samples: 9912 genes: 11403
train: 7929 val: 1983 tumor_frac: 0.9266545601291364


In [ ]:
# === 4) Build schedules from saved graphs (depth for GNNs) ===
# ----- Layered schedule (padded feedforward constraint) -----
node_layered = pd.read_csv(LAYERED_GRAPH_DIR / "node_table_layered.csv")
edge_layered = pd.read_csv(LAYERED_GRAPH_DIR / "edge_table_layered.csv")
edge_index_layered = torch.load(LAYERED_GRAPH_DIR / "edge_index_layered.pt").long()

# Roots at the maximum layer (one copy per original root pathway)
root_path = LAYERED_GRAPH_DIR / "root_pathway_idx.pt"
if root_path.exists():
    root_ids_layered = torch.load(root_path).long().tolist()
else:
    root_ids_layered = (
        node_layered.query("node_type=='pathway'")
        .groupby("orig_id")["layered_id"]
        .max()
        .astype(int)
        .tolist()
    )

layered_schedule = build_layered_schedule(
    edge_index=edge_index_layered,
    node_layer=node_layered["layer"].to_numpy(dtype=np.int64),
    edge_type=edge_layered["edge_type"].astype(str).tolist(),
    gene_ids=node_layered.query("node_type=='gene' and layer==0")["layered_id"].astype(int).tolist(),
    root_ids=root_ids_layered,
)

L_BINN = int(layered_schedule.L)
print("Layered schedule | N:", layered_schedule.N, "E:", layered_schedule.E, "L:", L_BINN, "roots:", layered_schedule.root_ids.numel())

# ----- Raw DAG schedule (no padding) -----
node_raw = pd.read_csv(RAW_GRAPH_DIR / "node_table.csv")
edge_table_raw = pd.read_csv(RAW_GRAPH_DIR / "edge_table.csv")
edge_index_raw = torch.load(RAW_GRAPH_DIR / "edge_index.pt").long()

num_genes_raw = int((node_raw["node_type"]=="gene").sum())
pw = edge_table_raw.query("edge_type=='pathway_to_parent'")[["src","dst"]].astype(int)
root_ids_raw = sorted(set(pw["dst"].tolist()) - set(pw["src"].tolist()))

depth_schedule = build_depth_schedule(
    edge_index=edge_index_raw,
    gene_ids=list(range(num_genes_raw)),
    root_ids=root_ids_raw,
)
print("Raw DAG schedule | N:", depth_schedule.N, "E:", depth_schedule.E, "D:", depth_schedule.D, "roots:", len(root_ids_raw))

if depth_schedule.D != L_BINN:
    print(f"WARNING: raw DAG depth (D={depth_schedule.D}) != layered depth (L={L_BINN}).")
    print("We still run both; interpret depth-matching claims accordingly.")


Layered schedule | N: 16283 E: 44955 L: 12 roots: 29


/tmp/ipykernel_1879961/2675204810.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  edge_index_layered = torch.load(LAYERED_GRAPH_DIR / "edge_index_layered.pt").long()
/tm

Raw DAG schedule | N: 14201 E: 140581 D: 12 roots: 29


In [ ]:
# === 5) Prepare mapping/pathways for OFFICIAL BINN package (from our saved graph) ===
# The BINN package expects:
# - mapping_df columns: [input, translation]  (gene -> pathway)
# - pathways_df columns: [source, target]     (child pathway -> parent pathway)

# Pathway hierarchy edges (child -> parent) using names (Reactome IDs)
pw_edges = edge_table_raw.query("edge_type=='pathway_to_parent'")[["src_name","dst_name"]].copy()
pw_edges = pw_edges.rename(columns={"src_name":"source", "dst_name":"target"}).drop_duplicates()

# Leaf pathways in child->parent graph are those with indegree==0 (no children)
parents = set(pw_edges["target"].astype(str))
children = set(pw_edges["source"].astype(str))
leaf_pathways = sorted(children - parents)

# Gene->pathway edges, filtered to leaf pathways (matches the layered-graph construction)
g2p_edges = edge_table_raw.query("edge_type=='gene_to_pathway'")[["src_name","dst_name"]].copy()
g2p_edges = g2p_edges.rename(columns={"src_name":"input", "dst_name":"translation"}).drop_duplicates()
g2p_edges = g2p_edges[g2p_edges["translation"].astype(str).isin(leaf_pathways)].copy()

print("pathways_df edges:", len(pw_edges))
print("leaf pathways:", len(leaf_pathways))
print("mapping_df (gene->leaf pathway) edges:", len(g2p_edges))
print("mapped genes:", g2p_edges["input"].nunique())


pathways_df edges: 2814
leaf pathways: 1971
mapping_df (gene->leaf pathway) edges: 40059
mapped genes: 10138


In [ ]:
# === 6) Instantiate OFFICIAL BINN model (Hartman package) and align gene order for ALL models ===
# The BINN package defines an internal *input ordering* (binn.inputs).
# We enforce that ordering for every baseline, so they all see the same X.

# Minimal datamatrix: BINN only needs the entity column to build the network
data_matrix_min = pd.DataFrame({"Gene": expr.index.astype(str)})

binn_ref = BINN(
    data_matrix=data_matrix_min,
    mapping=g2p_edges,
    pathways=pw_edges,
    entity_col="Gene",
    input_col="input",
    translation_col="translation",
    source_col="source",
    target_col="target",
    activation="tanh",
    n_layers=L_BINN,
    n_outputs=2,
    dropout=0.0,          # dropout will be set per-run in the sweep
    heads_ensemble=False,
    device="cpu",         # force CPU at construction; we'll move to GPU later
)

binn_inputs = list(binn_ref.inputs)
print("BINN inputs:", len(binn_inputs), "genes (example):", binn_inputs[:5])

# Reorder expression rows to match BINN's internal input ordering
expr = expr.loc[binn_inputs, :].copy()

# Build X, y tensors
X_gene = torch.tensor(expr.to_numpy(dtype=np.float32).T)  # samples x genes
y = torch.tensor(y_np, dtype=torch.long)

print("X_gene:", tuple(X_gene.shape), "y:", tuple(y.shape))



[INFO] BINN is on device: cpu
BINN inputs: 10138 genes (example): ['ENSG00000000419', 'ENSG00000000460', 'ENSG00000000938', 'ENSG00000000971', 'ENSG00000001036']
X_gene: (9912, 10138) y: (9912,)


In [ ]:
# === 7) Build gene maps (now that gene order is fixed) ===
# expression gene order -> layered gene node ids
gene_rows = node_layered.query("node_type=='gene' and layer==0")[["orig_name","layered_id"]]
gene_to_layered = dict(zip(gene_rows["orig_name"].astype(str), gene_rows["layered_id"].astype(int)))
gene_map_layered = torch.tensor([gene_to_layered[g] for g in expr.index.astype(str)], dtype=torch.long)

# expression gene order -> raw gene node ids (0..num_genes_raw-1)
raw_gene_names = node_raw.query("node_type=='gene'")["node_name"].astype(str).tolist()
raw_gene_to_id = {g:i for i,g in enumerate(raw_gene_names)}
gene_map_raw = torch.tensor([raw_gene_to_id[g] for g in expr.index.astype(str)], dtype=torch.long)

print("gene_map_layered:", tuple(gene_map_layered.shape), "gene_map_raw:", tuple(gene_map_raw.shape))


gene_map_layered: (10138,) gene_map_raw: (10138,)


In [ ]:
# === 8) Train/val tensors + optional standardization (train-only fit) ===
X_train = X_gene[train_idx].cpu().numpy()
X_val   = X_gene[val_idx].cpu().numpy()

y_train = y[train_idx].to(device)
y_val   = y[val_idx].to(device)

if STANDARDIZE_X:
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)

X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
X_val   = torch.tensor(X_val, dtype=torch.float32, device=device)

print("X_train:", tuple(X_train.shape), "X_val:", tuple(X_val.shape))


X_train: (7929, 10138) X_val: (1983, 10138)


In [ ]:
# === 9) Shared training loop (same split, same optimizer, same loss) ===
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=512, shuffle=False)

def eval_auc(model: nn.Module) -> tuple[float, float]:
    model.eval()
    probs = []
    ys = []
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            p1 = torch.softmax(logits, dim=1)[:, 1]
            probs.append(p1.detach().cpu().numpy())
            ys.append(yb.detach().cpu().numpy())
    probs = np.concatenate(probs)
    ys = np.concatenate(ys)
    auc = roc_auc_score(ys, probs)
    pred = (probs >= 0.5).astype(np.int64)
    bacc = balanced_accuracy_score(ys, pred)
    return float(auc), float(bacc)

def train_one(model: nn.Module, *, lr: float, weight_decay: float, epochs: int, early_stop: int, name: str) -> dict:
    model = model.to(device)

    # class weights (simple inverse frequency)
    counts = torch.bincount(y_train)
    w = (1.0 / counts.float()).to(device)
    crit = nn.CrossEntropyLoss(weight=w)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_auc = -1.0
    best_state = None
    patience = 0

    for ep in range(1, epochs + 1):
        model.train()
        running = 0.0
        n = 0

        for xb, yb in train_loader:
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()

            running += float(loss.item()) * xb.size(0)
            n += xb.size(0)

        val_auc, val_bacc = eval_auc(model)

        if val_auc > best_auc + 1e-6:
            best_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1

        print(f"{name} | ep {ep:03d} | train_loss {running/max(1,n):.4f} | val_auc {val_auc:.4f} | val_bacc {val_bacc:.4f}")

        if patience >= early_stop:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_auc, val_bacc = eval_auc(model)

    n_params = sum(p.numel() for p in model.parameters())
    return {"val_auc": val_auc, "val_bacc": val_bacc, "n_params": int(n_params)}


In [ ]:
# === 10) Model factories (depth-matched to BINN) ===
import collections
import torch.nn as nn

def _remove_batchnorm_from_sequential(seq: nn.Sequential) -> nn.Sequential:
    """Drop BatchNorm1d modules from an nn.Sequential (keeps layer order otherwise)."""
    items = []
    for name, mod in seq.named_children():
        if isinstance(mod, nn.BatchNorm1d):
            continue
        items.append((name, mod))
    return nn.Sequential(collections.OrderedDict(items))

def make_official_binn(*, dropout: float, remove_batchnorm: bool) -> nn.Module:
    # Rebuild with the SAME mapping/pathways, but set dropout for this run.
    model = BINN(
        data_matrix=data_matrix_min,
        mapping=g2p_edges,
        pathways=pw_edges,
        entity_col="Gene",
        input_col="input",
        translation_col="translation",
        source_col="source",
        target_col="target",
        activation="tanh",
        n_layers=L_BINN,
        n_outputs=2,
        dropout=float(dropout),
        heads_ensemble=False,
        device="cpu",
    )
    if remove_batchnorm and isinstance(model.layers, nn.Sequential):
        model.layers = _remove_batchnorm_from_sequential(model.layers)
    return model

def make_models(*, dropout: float, d_vector: int) -> dict[str, nn.Module]:
    return {
        # Official BINN package baseline (Hartman et al.) as-is
        "layered_binn_exact_d1": make_official_binn(dropout=dropout, remove_batchnorm=False),

        # Same official BINN, but with BatchNorm removed (helps isolate BN effects)
        "layered_binn_pkg_no_bn": make_official_binn(dropout=dropout, remove_batchnorm=True),

        # GNN-style ablations on the padded layered graph (same depth L_BINN)
        "layered_gcn_shared_d1": GCNSharedD1(schedule=layered_schedule, gene_map=gene_map_layered, dropout=dropout),
        "layered_gat_gate_d1":   GATGateD1(schedule=layered_schedule, gene_map=gene_map_layered, dropout=dropout),
        "layered_vector_shared": VectorSharedD(schedule=layered_schedule, gene_map=gene_map_layered, d=int(d_vector), dropout=dropout),

        # Native DAG propagation baseline (same max depth, but no padding/copy nodes)
        "dag_binn_exact_d1": DAGBINNExactD1(schedule=depth_schedule, gene_map=gene_map_raw, dropout=dropout),

        # Our exact MPNN re-implementation on the padded graph (unique edge weights, scalar)
        # (No BatchNorm, so do not expect identical training curves vs official BINN.)
        "layered_binn_mpn_exact_d1": BINNExactD1(schedule=layered_schedule, gene_map=gene_map_layered, dropout=dropout, learn_padding=False),

        # Dense baseline
        "dense_mlp": DenseMLPBaseline(in_dim=X_gene.shape[1], dropout=dropout),
    }


In [ ]:
# === 11) Hyperparameter sweep (shared grid) ===
from itertools import product

def grid_iter(grid: dict):
    keys = list(grid.keys())
    for values in product(*[grid[k] for k in keys]):
        yield dict(zip(keys, values))

all_results = []

for cfg in grid_iter(GRID):
    set_seed(SEED)  # keep model init comparable across cfgs

    lr = float(cfg["lr"])
    wd = float(cfg["weight_decay"])
    dropout = float(cfg["dropout"])
    d_vector = int(cfg["d_vector"])

    print("\n============================================================")
    print("CONFIG:", cfg)
    print("============================================================")

    models = make_models(dropout=dropout, d_vector=d_vector)

    for name, model in models.items():
        print("\n---", name, "---")
        metrics = train_one(
            model,
            lr=lr,
            weight_decay=wd,
            epochs=EPOCHS,
            early_stop=EARLY_STOPPING,
            name=name,
        )
        row = {"model": name, **cfg, **metrics}
        all_results.append(row)

df = pd.DataFrame(all_results)
df.sort_values(["val_auc","val_bacc"], ascending=False).head(30)



CONFIG: {'lr': 0.001, 'weight_decay': 1e-05, 'dropout': 0.0, 'd_vector': 16}

[INFO] BINN is on device: cpu

[INFO] BINN is on device: cpu

--- layered_binn_exact_d1 ---
layered_binn_exact_d1 | ep 001 | train_loss 0.3476 | val_auc 0.9878 | val_bacc 0.9452
layered_binn_exact_d1 | ep 002 | train_loss 0.1729 | val_auc 0.9908 | val_bacc 0.9527
layered_binn_exact_d1 | ep 003 | train_loss 0.1084 | val_auc 0.9907 | val_bacc 0.9621
layered_binn_exact_d1 | ep 004 | train_loss 0.0679 | val_auc 0.9959 | val_bacc 0.9719
layered_binn_exact_d1 | ep 005 | train_loss 0.0511 | val_auc 0.9943 | val_bacc 0.9697
layered_binn_exact_d1 | ep 006 | train_loss 0.0364 | val_auc 0.9892 | val_bacc 0.9535
layered_binn_exact_d1 | ep 007 | train_loss 0.0338 | val_auc 0.9923 | val_bacc 0.9626
layered_binn_exact_d1 | ep 008 | train_loss 0.0361 | val_auc 0.9937 | val_bacc 0.9689
layered_binn_exact_d1 | ep 009 | train_loss 0.0277 | val_auc 0.9944 | val_bacc 0.9728
layered_binn_exact_d1 | ep 010 | train_loss 0.0180 | va

,model,lr,weight_decay,dropout,d_vector,val_auc,val_bacc,n_params
31,dense_mlp,0.0003,0.00001,0.2,16,0.996938,0.971510,2612162
15,dense_mlp,0.0010,0.00001,0.2,16,0.996916,0.975502,2612162
23,dense_mlp,0.0003,0.00001,0.0,16,0.996158,0.972693,2612162
7,dense_mlp,0.0010,0.00001,0.0,16,0.996081,0.970693,2612162
13,dag_binn_exact_d1,0.0010,0.00001,0.2,16,0.996030,0.982038,154842
0,layered_binn_exact_d1,0.0010,0.00001,0.0,16,0.995873,0.971877,49328423
29,dag_binn_exact_d1,0.0003,0.00001,0.2,16,0.995610,0.981766,154842
6,layered_binn_mpn_exact_d1,0.0010,0.00001,0.0,16,0.995599,0.979678,61298
21,dag_binn_exact_d1,0.0003,0.00001,0.0,16,0.995137,0.975053,154842
5,dag_binn_exact_d1,0.0010,0.00001,0.0,16,0.995006,0.978590,154842


In [ ]:
# === 12) Summary tables ===
# Best config per model
best_by_model = (
    df.sort_values(["model", "val_auc"], ascending=[True, False])
      .groupby("model", as_index=False)
      .head(1)
      .sort_values("val_auc", ascending=False)
)
best_by_model


,model,lr,weight_decay,dropout,d_vector,val_auc,val_bacc,n_params
31,dense_mlp,0.0003,0.00001,0.2,16,0.996938,0.971510,2612162
13,dag_binn_exact_d1,0.0010,0.00001,0.2,16,0.996030,0.982038,154842
0,layered_binn_exact_d1,0.0010,0.00001,0.0,16,0.995873,0.971877,49328423
6,layered_binn_mpn_exact_d1,0.0010,0.00001,0.0,16,0.995599,0.979678,61298
4,layered_vector_shared,0.0010,0.00001,0.0,16,0.959165,0.891524,264562
17,layered_binn_pkg_no_bn,0.0003,0.00001,0.0,16,0.516877,0.500000,49293551
3,layered_gat_gate_d1,0.0010,0.00001,0.0,16,0.500000,0.500000,16379
2,layered_gcn_shared_d1,0.0010,0.00001,0.0,16,0.500000,0.500000,16355


In [ ]:
# Optional: save results for tomorrow's report
out_csv = OUT_DIR / "tcga_binn_vs_gnn_results.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


Saved: /home/maani/niloo/binn_gnn_repo_ready/binn_gnn_repo_ready/outputs/tcga_binn_vs_gnn_results.csv
